# K3b — Cross-Encoder LoRA Fine-Tune + Eval Gate (spec 53_K3b)

LoRA fine-tunes `BAAI/bge-reranker-v2-m3` on denoised hard negatives sampled from the fused
candidate pool, using a grouped listwise-softmax loss weighted by goal-progress labels. The adapter
is evaluated as a final-stage reranker stacked on top of K2 (`ChainReranker(k2, k3b)`).

**Cross-encoder only.** This notebook does NOT train any non-CE model. K2 (LGBM LambdaMART) is a
PREREQUISITE: train + save it in its own notebook (`phase2_rerank.ipynb` → `k2_lgbm.txt`) first; here
it is only LOADED to provide the baseline + the slice the CE re-scores. Retrieval/fusion is the
candidate generator (not a reranker model).

**Architecture:** [prereq] K2 (loaded) → K3b (bge-reranker-v2-m3 + LoRA, top-K re-score).
**Run on:** Colab T4/G4 (16 GB). **Spec:** `.claude/documents/features/53_K3b_ce_lora_finetune.md`.

## 1. Drive + HF auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'
OUT = f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(OUT, exist_ok=True)
try:
    t = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = os.environ['HUGGINGFACE_HUB_TOKEN'] = t
    from huggingface_hub import login
    login(t)
    print('HF ok')
except Exception as e:
    print('no HF_TOKEN secret:', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
# peft + trackio for K3b (LoRA adapter + loss logging).
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas peft trackio
# Colab preinstalls torchao 0.10, which trips peft's is_torchao_available() version check
# (peft >=0.19 requires torchao >0.16 and RAISES on an older one). LoRA does not use torchao, so
# remove it — avoids "Found an incompatible version of torchao". (On a re-run mid-session, also
# Restart Runtime once after this so the already-imported torchao 0.10 is cleared.)
!pip -q uninstall -y torchao
import sys
sys.path.insert(0, '.')

## 3. Config

T4/G4 runtime levers are surfaced here with prominent comments. Tune these first when adjusting
the memory/speed/quality trade-off before touching deeper parameters.

In [ ]:
# ── Cross-encoder model + retrieval budget ──────────────────────────────────
CE_MODEL = 'BAAI/bge-reranker-v2-m3'  # bge-reranker-v2-m3: 568M XLM-R backbone
CROSS_ENCODER_K = 100   # candidates scored per turn at both train and serve
N_NEG = 15              # hard negatives sampled per gold (contrast knob)
K_MIN = 4               # minimum negatives required to keep a training turn
SEED = 0

# ── Sequence / dtype (train==serve — do NOT split these) ─────────────────────
MAX_LEN = 2048          # ← T4/G4: must match at serve; lower to 1536 if OOM after A1 re-measure
MAX_DOC_TOK = 1100      # doc-side token cap (query preserved); pair = query + doc + 4 specials
DTYPE = 'auto'          # 'auto' → fp16 on T4 (no bf16), bf16 on Ampere+; 'fp16'/'bf16' to force

# ── Training-data subset ─────────────────────────────────────────────────────
# Caps train SESSIONS for the cross-encoder fine-tune. Full train is 15,199 sessions; 7600 ≈ half.
# Raise toward 15199 on a faster GPU for full coverage; lower to 1500–2000 if OOM / time-tight.
TRAIN_SUBSET = 4000

# Dev/test sessions used for the eval gate (cell 7). Full test split = 1000 sessions.
DEV_SUBSET = 400

# ── Val nDCG@20 callback subset ──────────────────────────────────────────────
# Train-internal held-out turns for the real nDCG@20 early-stopping callback (spec §4.5/§4.7).
# Bounds per-epoch eval cost; 500 turns ≈ 2-3 min/epoch on T4 at CROSS_ENCODER_K=100.
VAL_NDCG_TURNS = 500

# ── LoRA config ──────────────────────────────────────────────────────────────
LORA = {
    'r': 16,                          # rank: 8 = memory-light, 16 = standard (our default)
    'alpha': 32,                      # scaling = alpha/r = 2 (common default)
    'dropout': 0.05,
    'target_modules': ['query', 'value'],   # XLM-R attention projections
}

# ── Training loop ─────────────────────────────────────────────────────────────
# T4/G4 tuning: if OOM lower batch_groups (→1) or MAX_LEN (→1536 after A1 re-measure).
# Effective batch = batch_groups * grad_accum = 2 * 16 = 32 groups (stability without OOM at seq=2048).
TRAIN = {
    'epochs': 2,                   # upper bound; early-stop (patience=1) picks the real stop
    'lr': 1e-4,
    'weight_decay': 0.0,
    'batch_groups': 2,             # ← micro-batch size; lower first if OOM
    'grad_accum': 16,              # gradient-accumulation steps → effective batch = 2*16 = 32
    'warmup': 0.05,                # fraction of total optimizer steps for linear warmup
    'log_every': 50,               # log to Trackio every N optimizer steps
    'group_by_length': True,       # length-grouped sampler (less pad waste on T4)
    'early_stop_patience': 1,      # stop after N epochs without val nDCG@20 improvement
}

# ── Retrieval + dense model (shared with the K2 prerequisite setup) ──────────
TOPK = 500
DENSE_MODEL = 'BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
CONTENT_MODALITIES = {
    'cknn_audio': 'audio-laion_clap',
    'cknn_attr': 'attributes-qwen3_embedding_0.6b',
}
ORG = 'talkpl-ai'
import glob
ENRICHED_GLOB = f'{OUT}/catalog_enriched_*.parquet'

print('Config OK — CE_MODEL:', CE_MODEL, '| MAX_LEN:', MAX_LEN, '| DTYPE:', DTYPE,
      '| TRAIN_SUBSET:', TRAIN_SUBSET, '| VAL_NDCG_TURNS:', VAL_NDCG_TURNS)

## 4. Load data + catalog (enriched) + channels + RRFFusion + train K2

Reuses the phase2_rerank setup: Conversations, Catalog from enriched parquet, channels, fusion,
FeatureBuilder + dense_cos, LGBMReranker.fit. K3b stacks on top of K2 so K2 must be trained first.
Asserts 100% enriched coverage over the candidate pool (spec §2).

In [ ]:
import os, glob, pickle, hashlib
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion
from mcrs.rerank.features import FeatureBuilder
from mcrs.rerank.lgbm import LGBMReranker

# This notebook is the CROSS-ENCODER stage. It does NOT train any non-CE model: K2 (LGBM) is a
# PREREQUISITE produced by its own notebook (phase2_rerank.ipynb) and LOADED here. Retrieval/fusion
# is the candidate generator (not a reranker model). The CE stacks on the loaded K2 (ChainReranker).

# ── Catalog (enriched from A1 parquet) ──────────────────────────────────────
meta_rows = load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
_enr_files = sorted(glob.glob(ENRICHED_GLOB))
assert _enr_files, f'No enriched parquet found at {ENRICHED_GLOB} — run A1 notebook first'
_edf = pd.read_parquet(_enr_files[-1])
enr = dict(zip(_edf['track_id'], _edf['enriched_doc']))
cat = Catalog(meta_rows, enriched_docs=enr)
USE_ENRICHED = True
print(f'enriched docs: {len(enr)} (from {_enr_files[-1].split("/")[-1]})')

# ── Spec §2: assert 100% enriched coverage (hard fail — K3b needs enriched docs everywhere) ──
_sample_pool_tids = list(cat._meta.keys())[:500]
assert all(cat.is_enriched(t) for t in _sample_pool_tids), \
    'K3b requires 100% enriched doc coverage over the catalog (spec §2)'
print('enriched coverage check: OK')

# ── Track/User embeddings ──────────────────────────────────────────────────
tre = load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings', split='all_tracks')
_avail = set(tre.column_names)
CKNN_MODS = {lab: mod for lab, mod in CONTENT_MODALITIES.items() if mod in _avail}
te = {lab: TrackEmbeddings(tre.select_columns(['track_id', mod]), modalities=[mod])
      for lab, mod in CKNN_MODS.items()}
te_cf = TrackEmbeddings(tre.select_columns(['track_id', 'cf-bpr']), modalities=['cf-bpr'])
ued = load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings')
ue = UserEmbeddings([r for sp in ued for r in ued[sp]])

# ── Conversations ──────────────────────────────────────────────────────────
dsd = load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')
_n_tr = min(TRAIN_SUBSET, len(dsd['train']))
conv_tr = Conversations(dsd['train'].select(range(_n_tr)))
_n_dv = min(DEV_SUBSET, len(dsd['test']))
conv_dv = Conversations(dsd['test'].select(range(_n_dv)))
print(f'sessions: train={_n_tr}, dev={_n_dv}')

# ── Dense model + doc matrix (CACHED: encoding ~47k docs is a fixed per-session cost) ──
model = SentenceTransformer(DENSE_MODEL, device='cuda')
_dm_key = hashlib.md5(f'{DENSE_MODEL}|{os.path.basename(_enr_files[-1])}'.encode()).hexdigest()[:10]
_DOC_MAT_NPY = f'{OUT}/doc_mat_{_dm_key}.npy'
if os.path.exists(_DOC_MAT_NPY):
    doc_mat = np.load(_DOC_MAT_NPY)
    print(f'loaded cached doc_mat <- {_DOC_MAT_NPY}  shape={doc_mat.shape}')
else:
    doc_mat = model.encode(
        [cat.id_to_metadata(t, enriched=USE_ENRICHED) for t in cat.index_to_id],
        batch_size=256, normalize_embeddings=True, show_progress_bar=True,
    )
    np.save(_DOC_MAT_NPY, doc_mat)
    print(f'built + cached doc_mat -> {_DOC_MAT_NPY}  shape={doc_mat.shape}')

# ── Related-artist co-occurrence (cached) ──────────────────────────────────
COOC_PKL = f'{OUT}/artist_cooc.pkl'
if os.path.exists(COOC_PKL):
    cooc = pickle.load(open(COOC_PKL, 'rb'))
    print('loaded cooc', len(cooc), 'artists')
else:
    cooc = build_artist_cooc(dsd['train'], tid_to_artists_from_catalog(cat))
    pickle.dump(cooc, open(COOC_PKL, 'wb'))
    print('built cooc', len(cooc), 'artists')

# ── Channels + fusion (the candidate generator the cross-encoder reranks) ──
dense = DenseChannel(
    cat.index_to_id, doc_mat,
    lambda qs: model.encode([DENSE_QUERY_PREFIX + q for q in qs],
                             batch_size=256, normalize_embeddings=True),
    normalize=False,
)
cknn = [ContentKNNChannel(te[lab], mod, label=lab) for lab, mod in CKNN_MODS.items()]
chans = [
    BM25Channel(cat, enriched=USE_ENRICHED),
    dense,
    *cknn,
    CFChannel(ue, te_cf, 'cf-bpr'),
    SameArtistChannel(cat),
    RelatedArtistChannel(cat, cooc),
]
fusion = RRFFusion(chans, k=60)
labels = [c.label for c in chans]
print('channels:', labels)

# ── K2 feature pipeline (needed only to RUN the loaded K2 — NOT to train it) ──
# These MUST match what phase2_rerank.ipynb used to train K2 (same channel set + dense_cos feature),
# or the loaded booster's feature columns won't line up. Plain QueryBuilder for retrieval.
qb_plain = QueryBuilder()
_qcache = {}

def precompute_qvecs(turns):
    texts = [DENSE_QUERY_PREFIX + qb_plain.build(t).text for t in turns]
    mat = model.encode(texts, batch_size=256, normalize_embeddings=True)
    _qcache.update({(t.session_id, t.turn_number): mat[i] for i, t in enumerate(turns)})

def dense_cos(ctx, tid):
    j = cat.id_to_index.get(tid)
    if j is None:
        return 0.0
    v = _qcache.get((ctx.session_id, ctx.turn_number))
    if v is None:
        v = model.encode([DENSE_QUERY_PREFIX + qb_plain.build(ctx).text], normalize_embeddings=True)[0]
        _qcache[(ctx.session_id, ctx.turn_number)] = v
    return float(v @ doc_mat[j])

fb = FeatureBuilder(cat, labels, score_fns={'dense_cos': dense_cos})

# ── LOAD K2 (prerequisite from phase2_rerank.ipynb) — do NOT train it here ──
K2_TXT = f'{OUT}/k2_lgbm.txt'
assert os.path.exists(K2_TXT), (
    f'Prerequisite K2 model not found at {K2_TXT}. This is the cross-encoder notebook and it does '
    f'NOT train K2 — run phase2_rerank.ipynb first to train + save K2 (it writes k2_lgbm.txt), then '
    f're-run this cell.')
k2 = LGBMReranker(fb, n_estimators=500, neg_cap=150, early_stopping_rounds=50, val_fraction=0.1).load(K2_TXT)
print('loaded prerequisite K2 <-', K2_TXT)

tr = list(conv_tr.turns())
print(f'train turns: {len(tr)} | retrieval + loaded K2 ready (cross-encoder is trained next)')

## 5. Build enriched QueryBuilder + CE training groups

Builds `(query_text, [pos_doc, neg_doc...], group_weight)` triples using:
- Enriched `QueryBuilder(markers=True, taste_items=5)` for richer query context
- `build_ce_training_groups` with denoised hard negatives + goal-progress weighting
- `goal_progress_assessments` from the raw session row

In [ ]:
import os, pickle
from mcrs.retrieval.query import QueryBuilder
from mcrs.training.ce_data import build_ce_training_groups

# Enriched query builder: markers=True adds request:/context:/goal:/taste: sections;
# taste_items=5 appends the 5 most-recent liked tracks (newest-first) for warm sessions.
def _meta_str(tid, key):
    """Coerce a catalog metadata field to a string. Real Track-Metadata fields are LIST-valued
    (e.g. ['Bon Iver']) -> join them so the taste clause reads 'Bon Iver – Holocene', not "['..']"."""
    if tid not in cat._meta:
        return ''
    v = cat.metadata(tid).get(key)
    if isinstance(v, list):
        return ', '.join(str(x) for x in v)
    return '' if v is None else str(v)

def track_label(tid):
    if tid not in cat._meta:
        return None
    return f"{_meta_str(tid, 'artist_name')} – {_meta_str(tid, 'track_name')}"

qb_ce = QueryBuilder(markers=True, taste_items=5, track_label_fn=track_label)
# qb_plain (plain retrieval builder, bound in the K2/setup cell) is passed as fusion_query_builder
# so the candidate pool is fused with the same plain query InferenceHarness uses at serve, while
# qb_ce is the CE pair query only (train == serve parity, fix m3).

# goal_progress_assessments: list of {turn_number, goal_progress_assessment} per session row.
_gp_lookup = {}
for sid, row in conv_tr._rows.items():
    _gp_lookup[sid] = {
        int(a['turn_number']): a['goal_progress_assessment']
        for a in (row.get('goal_progress_assessments') or [])
    }

def gp_fn(turn):
    """Goal-progress label for this turn, or None if missing."""
    return _gp_lookup.get(turn.session_id, {}).get(turn.turn_number)

# ── CACHE ──────────────────────────────────────────────────────────────────────────────────────
# Building groups runs FUSION over every train turn (the slow part) + build_doc. Cache (groups,
# report) to Drive so a session reconnect doesn't re-run it. The key encodes the config that changes
# the groups (subset/K/N/k_min/seed) AND the enriched-corpus id (_dm_key, from cell 4) so a new A1
# parquet self-invalidates the cache. Still DELETE the .pkl if you change the query format
# (qb_ce/taste) or the channel set — those are not captured by the key.
_GROUPS_PKL = f'{OUT}/ce_groups_sub{TRAIN_SUBSET}_k{CROSS_ENCODER_K}_n{N_NEG}_kmin{K_MIN}_seed{SEED}_{_dm_key}.pkl'
if os.path.exists(_GROUPS_PKL):
    with open(_GROUPS_PKL, 'rb') as f:
        groups, report = pickle.load(f)
    print(f'loaded cached groups <- {_GROUPS_PKL}  ({len(groups)} groups)')
else:
    report = {}
    groups = build_ce_training_groups(
        qb_ce, fusion, tr,
        lambda t: conv_tr.gold(t.session_id, t.turn_number),
        catalog=cat,
        cross_encoder_k=CROSS_ENCODER_K,
        n_negatives=N_NEG,
        k_min=K_MIN,
        seed=SEED,
        gp_fn=gp_fn,
        fusion_query_builder=qb_plain,   # FIX m3: fuse pool with plain query (matches serve)
        fusion_chunk=1000, show_progress=True,   # chunked fusion -> tqdm progress bar + bounded memory
        report=report,
    )
    with open(_GROUPS_PKL, 'wb') as f:   # docs are full strings -> file can be large; lives on Drive
        pickle.dump((groups, report), f)
    print(f'built + cached groups -> {_GROUPS_PKL}')

assert 'kept_keys' in report, 'stale groups cache (missing kept_keys) — delete the ce_groups_*.pkl and re-run'
print('build_ce_training_groups report:', {k: v for k, v in report.items() if k != 'kept_keys'})
# report keys: dropped_no_gold, dropped_few_neg, kept, kept_keys (1:1 with groups)
print(f'groups built: {len(groups)} | example query[:80]: {groups[0][0][:80] if groups else "(empty)"}')

## 6. Session-disjoint train/val split + Trackio init + LoRA fine-tune

Uses fold 0 as the validation set (10% of sessions). `val_eval_fn=make_val_ndcg` wires the
real nDCG@20 ranking metric as the early-stopping signal (spec §4.5/§4.7): a closure over the
IN-TRAINING model scores a fixed `dev[:VAL_NDCG_TURNS]` subset per epoch via ChainReranker(k2, k3_tmp),
bounded to `VAL_NDCG_TURNS` turns to cap per-epoch cost. The best-checkpoint is the epoch with the
highest val nDCG@20, not the lowest val loss.

The trained adapter is pushed to HF Hub by revision for provenance.

In [ ]:
import os
import torch
from mcrs.training.ce_data import assign_session_folds
from mcrs.training.ce_finetune import finetune_cross_encoder
from mcrs.rerank.cross_encoder import doc_token_budget, truncate_doc_tokens
from mcrs.rerank.neural import NeuralReranker, ChainReranker
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import InferenceHarness
from mcrs.eval.harness import GoldRow
from mcrs.eval.official import score_official

# ── Session-disjoint 10-fold split over KEPT groups (FIX M1) ─────────────────
# report['kept_keys'] is 1:1 with groups (covers both no-gold AND few-neg drops).
# Derive all fold assignments from it so len(group_folds) == len(groups) always.
kept_keys = report['kept_keys']   # list of (session_id, turn_number), 1:1 with groups
assert len(kept_keys) == len(groups), \
    f'kept_keys/group mismatch: {len(kept_keys)} vs {len(groups)}'

group_folds = assign_session_folds(
    [sid for (sid, _t) in kept_keys], k=10, seed=SEED
)   # 1:1 with groups and kept_keys
assert len(group_folds) == len(groups), \
    f'fold/group mismatch: {len(group_folds)} vs {len(groups)}'

tr_groups = [g for g, f in zip(groups, group_folds) if f != 0]
va_groups = [g for g, f in zip(groups, group_folds) if f == 0]
print(f'train groups: {len(tr_groups)} | val groups: {len(va_groups)}')
# Guard: an empty val fold would crash make_val_ndcg in score_official.
assert len(va_groups) > 0, 'val fold is empty — lower K_MIN or raise TRAIN_SUBSET'

# ── FIX C1: TRAIN-internal validation set for early stopping ─────────────────
# Spec §4.6: dev must stay clean; early stopping uses an in-TRAIN held-out fold.
# Val turns = kept turns whose session is in fold 0 (session-disjoint from the
# train groups above — guaranteed by assign_session_folds).
_val_fold_sids = {sid for (sid, _t), f in zip(kept_keys, group_folds) if f == 0}
_sid_to_turn = {(t.session_id, t.turn_number): t for t in tr}
val_turns_tr = [
    _sid_to_turn[(sid, tn)]
    for (sid, tn) in kept_keys
    if sid in _val_fold_sids
][:VAL_NDCG_TURNS]   # cap at VAL_NDCG_TURNS to bound per-epoch eval cost

_golds_val_tr = [
    GoldRow(t.session_id, t.user_id, t.turn_number,
            conv_tr.gold(t.session_id, t.turn_number))
    for t in val_turns_tr
]
_asm_val = TopKAssembler(cat)

# Precompute query vecs for the train-val turns (dense_cos feature inside InferenceHarness)
precompute_qvecs(val_turns_tr)

# ── val nDCG@20 closure wired to the IN-TRAINING model (uses train-internal val) ──
# Spec §4.5/§4.7: real ranking metric drives best-checkpoint + early stopping.
# Points at val_turns_tr + their golds (NOT conv_dv/dv — dev stays clean until cell 7).
def make_val_ndcg(model, tok):
    """Return val nDCG@20 over val_turns_tr using the in-training model (spec §4.5/§4.7).
    Train-internal hold-out (fold 0 sessions); bounded to VAL_NDCG_TURNS for cost."""
    dev = next(model.parameters()).device
    use_amp = (str(dev).startswith('cuda'))
    try:
        amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    except Exception:
        amp_dtype = torch.float16

    enc = lambda s: tok.encode(s, add_special_tokens=False, truncation=True, max_length=MAX_LEN)

    def _score_fn(pairs):
        queries = [q for q, _ in pairs]
        docs = []
        for q, d in pairs:
            budget = doc_token_budget(len(enc(q)), MAX_LEN, MAX_DOC_TOK)
            docs.append(truncate_doc_tokens(enc, tok.decode, d, budget))
        feats = tok(queries, docs, padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors='pt')
        feats = {k: v.to(dev) for k, v in feats.items()}
        with torch.inference_mode():
            with torch.autocast(device_type=str(dev).split(':')[0], dtype=amp_dtype, enabled=use_amp):
                scores = model(**feats).logits.squeeze(-1)
        return scores.float().cpu().tolist()

    # ChainReranker(k2, k3_tmp) over the TRAIN-internal val subset (not dev)
    k3_tmp = NeuralReranker(cat, qb_ce, _score_fn, cross_encoder_k=CROSS_ENCODER_K, enriched=True)
    rows = InferenceHarness(qb_plain, fusion, _asm_val,
                            reranker=ChainReranker(k2, k3_tmp), topk=TOPK).run(val_turns_tr)
    return score_official(rows, _golds_val_tr, len(cat))['ndcg@20']

# Metrics print straight to the CELL OUTPUT — no Trackio dashboard (its 127.0.0.1 link is not
# reachable from your browser on Colab). train_log keeps the history for later inspection/plotting.
train_log = []

class _PrintLogger:
    def log(self, d, step=None):
        train_log.append({'step': step, **d})
        flat = '  '.join(f'{k}={round(v, 4) if isinstance(v, float) else v}' for k, v in d.items())
        print(f'[step {step}] {flat}', flush=True)

logger = _PrintLogger()
CKPT_DIR = f'{OUT}/ckpt_k3b'
os.makedirs(CKPT_DIR, exist_ok=True)

# val_eval_fn=make_val_ndcg: real nDCG@20 over VAL_NDCG_TURNS TRAIN-internal turns
# drives early stopping and best-checkpoint selection (spec §4.5/§4.7).
adapter = finetune_cross_encoder(
    tr_groups, va_groups,
    base_model=CE_MODEL,
    lora_cfg=LORA,
    max_length=MAX_LEN,
    max_doc_tokens=MAX_DOC_TOK,
    dtype=DTYPE,
    train_cfg=TRAIN,
    logger=logger,
    out_dir=CKPT_DIR,
    val_eval_fn=make_val_ndcg,
)
print('adapter saved to:', adapter)

# Push adapter to HF Hub by revision for provenance (spec §4.7).
if adapter:
    from huggingface_hub import HfApi
    _api = HfApi()
    HUB_ADAPTER_REPO = f'{ORG}/k3b-lora-adapter'
    try:
        _api.create_repo(HUB_ADAPTER_REPO, exist_ok=True, private=True)
        _api.upload_folder(
            folder_path=adapter,
            repo_id=HUB_ADAPTER_REPO,
            commit_message=f'k3b adapter TRAIN_SUBSET={TRAIN_SUBSET} MAX_LEN={MAX_LEN} LORA_R={LORA["r"]}',
        )
        print('pushed adapter to Hub:', HUB_ADAPTER_REPO)
    except Exception as e:
        print('Hub push skipped (ok in offline/test run):', e)

## 7. Eval gate: K2 vs K2+K3b on dev (overall + per-segment + per-goal-progress)

Ships only if `K2+K3ft > K2`. Also slices by segment (cold/warm) and by the gold turn's
goal-progress label. Abort the goal-progress lever if off-goal-gold nDCG regresses (spec §4.3).

In [ ]:
import json
from mcrs.rerank.cross_encoder import build_cross_encoder_score_fn
from mcrs.rerank.neural import NeuralReranker, ChainReranker
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import InferenceHarness, validate_submission
from mcrs.eval.harness import GoldRow
from mcrs.eval.official import score_official

# Load the fine-tuned cross-encoder score_fn (adapter merged into base for fast inference).
ce_score = build_cross_encoder_score_fn(
    CE_MODEL, device='cuda',
    max_length=MAX_LEN,
    max_doc_tokens=MAX_DOC_TOK,
    dtype=DTYPE,
    lora_adapter=adapter,   # local ckpt dir; or HUB_ADAPTER_REPO for Hub-loaded
)

# K3b reranker: uses the enriched QueryBuilder (same as training — train==serve)
k3b = NeuralReranker(
    cat, qb_ce, ce_score,
    cross_encoder_k=CROSS_ENCODER_K,
    enriched=True,
)

# Dev turns + gold rows
dv = list(conv_dv.turns())
precompute_qvecs(dv)   # batch encode dev queries into _qcache for dense_cos feature
golds = [
    GoldRow(t.session_id, t.user_id, t.turn_number,
            conv_dv.gold(t.session_id, t.turn_number))
    for t in dv
]
keys = [(g.session_id, g.turn_number) for g in golds]
asm = TopKAssembler(cat)

# Run K2 and K2+K3b inference
k2_rows = InferenceHarness(qb_plain, fusion, asm, reranker=k2, topk=TOPK).run(dv)
k3b_rows = InferenceHarness(qb_plain, fusion, asm, reranker=ChainReranker(k2, k3b), topk=TOPK).run(dv)
validate_submission(k2_rows, catalog=cat, expected_keys=keys)
validate_submission(k3b_rows, catalog=cat, expected_keys=keys)

s_k2 = score_official(k2_rows, golds, len(cat))
s_k3b = score_official(k3b_rows, golds, len(cat))
print(f'Overall nDCG@20:  K2={round(s_k2["ndcg@20"], 4)}  K2+K3b={round(s_k3b["ndcg@20"], 4)}')
delta = s_k3b['ndcg@20'] - s_k2['ndcg@20']
print(f'Delta: {round(delta, 4)} ({"SHIPS" if delta > 0 else "ABORT — no gain"})')

# Per-segment slice (cold/warm)
def _seg_score(rows, g_list, seg):
    gs = [g for g, t in zip(g_list, dv) if t.segment == seg]
    if not gs:
        return None
    ks = {(g.session_id, g.turn_number) for g in gs}
    rs = [r for r in rows if (r.session_id, r.turn_number) in ks]
    return round(score_official(rs, gs, len(cat))['ndcg@20'], 4)

for seg in ('cold', 'warm'):
    print(f'  {seg}: K2={_seg_score(k2_rows, golds, seg)}  K2+K3b={_seg_score(k3b_rows, golds, seg)}')

# Per-goal-progress label slice (from dev conversation rows)
_gp_dv = {}
for sid, row in conv_dv._rows.items():
    for a in (row.get('goal_progress_assessments') or []):
        _gp_dv[(sid, int(a['turn_number']))] = a['goal_progress_assessment']

def _gp_score(rows, g_list, label):
    gs = [g for g in g_list if _gp_dv.get((g.session_id, g.turn_number)) == label]
    if not gs:
        return None
    ks = {(g.session_id, g.turn_number) for g in gs}
    rs = [r for r in rows if (r.session_id, r.turn_number) in ks]
    return round(score_official(rs, gs, len(cat))['ndcg@20'], 4)

for lbl in ('MOVES_TOWARD_GOAL', 'DOES_NOT_MOVE_TOWARD_GOAL'):
    v_k2 = _gp_score(k2_rows, golds, lbl)
    v_k3b = _gp_score(k3b_rows, golds, lbl)
    print(f'  goal_progress={lbl}: K2={v_k2}  K2+K3b={v_k3b}')
    # Spec §4.3 guard: abort goal-progress lever if off-goal-gold nDCG regresses.
    if lbl == 'DOES_NOT_MOVE_TOWARD_GOAL' and v_k2 is not None and v_k3b is not None:
        if v_k3b < v_k2:
            print(f'  WARNING: off-goal-gold nDCG regressed {v_k3b} < {v_k2} — abort goal-progress lever (spec §4.3)')

# Persist results
_gate_result = {'k2': s_k2, 'k2_k3b': s_k3b, 'delta_ndcg20': delta,
                'ships': delta > 0, 'adapter': adapter}
json.dump(_gate_result, open(f'{OUT}/phase2_k3b_gate.json', 'w'), indent=2)
print('gate result saved ->', f'{OUT}/phase2_k3b_gate.json')
# Ships only if K2+K3b > K2; abort goal-progress lever if off-goal-gold nDCG regresses (spec §4.3)

## 8. Gate decision + next steps

**Ship criteria (spec §6):**
- `K2+K3b nDCG@20 > K2 nDCG@20` overall — otherwise revert to K2 alone.
- No regression on the `DOES_NOT_MOVE_TOWARD_GOAL` segment (§4.3 guard).

**If the gate passes:**
1. Retrain the cross-encoder on train+dev (blind run); the K2 prerequisite is retrained in its own notebook.
2. The adapter on HF Hub is the artifact to package in the CodaBench submission.
3. Next lever: larger `TRAIN_SUBSET` (toward full train) or higher `CROSS_ENCODER_K` on an A100.

**If the gate fails:**
- K3b gain is bounded by recall@pool — if pool quality is low, fix R7/retrieval first.
- Try reducing `N_NEG` (15→8) for harder contrast, or `CROSS_ENCODER_K` (100→50) to reduce noise.
- Check whether cold turns (0 history) dominate the regression — they have no `taste:` clause.

**Note:** OOF stacking of the CE score back into K2 is intentionally NOT in this notebook (it would
retrain a K2 variant). If you want it later, it belongs in the K2 notebook, consuming a CE-score
artifact exported from here.